### Load and Compute

In [1]:
import numpy as np
import pandas as pd
import os, glob
from pathlib import Path
from scipy.integrate import trapz

try:
    import utils.process_analysis

except ImportError as e:
    print(f"\nSTILL FAILING: {e}")

### Setup

In [2]:
DATA_DIR        = "../data/02_processed_combined_data"
CONDITIONS_FILE = "../data/sub_conditions.csv"
OUTPUT_DIR      = "../data/04_analysis/h3_hysteresis_results/all_subjects/"

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# Constants
FS             = 20
WINDOW_SECONDS = 15
STEP_SECONDS   = 1
WIN_SIZE       = int(WINDOW_SECONDS * FS)
STEP_SIZE      = int(STEP_SECONDS * FS)
BRANCH_POINT   = 205
TRANSITION_END = BRANCH_POINT + WINDOW_SECONDS # 220s

SEGMENTS_PC = {1: (9, 54), 2: (55, 205), 3: (205, 243), 4: (243, 278), 5: (279, 319)}

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']


### define one helper function

hardcoded here because otherwise too complex

In [4]:
# --- FUNC 1: CAUSAL (forward-looking) SLIDING WINDOW ---
def sliding_window_group_isc(df_pc, df_pcm, win_size, step_size, n_boot=200):
    n_samples = min(df_pc.shape[0], df_pcm.shape[0])
    # Limit to relevant movie length
    max_time = SEGMENTS_PC[5][1] * FS + (10 * FS)
    if n_samples > max_time: n_samples = int(max_time)
    
    n_pc, n_pcm = df_pc.shape[1], df_pcm.shape[1]
    results = []

    # Right-Aligned (Causal) Loop
    for start_idx in range(0, n_samples - win_size, step_size):
        end_idx = start_idx + win_size
        time_sec = end_idx / FS
        
        # Data Slices
        w_pc = df_pc.iloc[start_idx:end_idx, :].values
        w_pcm = df_pcm.iloc[start_idx:end_idx, :].values
        
        # Bootstrap
        boot_rs = []
        for _ in range(n_boot):
            i_pc = np.random.choice(n_pc, n_pc, replace=True)
            i_pcm = np.random.choice(n_pcm, n_pcm, replace=True)
            # Correlation of means
            r = np.corrcoef(w_pc[:, i_pc].mean(1), w_pcm[:, i_pcm].mean(1))[0,1]
            boot_rs.append(r)
        
        # Fisher Mean
        z_vals = np.arctanh(np.clip(boot_rs, -0.999, 0.999))
        mean_r = np.tanh(np.mean(z_vals))
        results.append({'Time_Sec': time_sec, 'Mean_r': mean_r})
        
    return pd.DataFrame(results)


# --- FUNC 2: DECAY QUANTIFICATION ---
def quantify_decay(df_timecourse):
    stats = []
    for sig in df_timecourse['Signal'].unique():
        sub = df_timecourse[df_timecourse['Signal'] == sig].sort_values('Time_Sec')
        
        # 1. Baseline (Pre-Branch)
        base_mask = (sub['Time_Sec'] >= (BRANCH_POINT - 15)) & (sub['Time_Sec'] <= BRANCH_POINT)
        baseline_r = sub.loc[base_mask, 'Mean_r'].mean()
        
        # 2. Decay Zone (205-220s)
        decay = sub[(sub['Time_Sec'] > BRANCH_POINT) & (sub['Time_Sec'] <= TRANSITION_END)]
        
        if decay.empty: continue
        
        # Metric A: Half-Life
        target = baseline_r * 0.5
        below = decay[decay['Mean_r'] < target]
        t_half = below['Time_Sec'].iloc[0] - BRANCH_POINT if not below.empty else WINDOW_SECONDS
        
        # Metric B: Persistence (Normalized AUC)
        auc = trapz(decay['Mean_r'], decay['Time_Sec'])
        max_area = baseline_r * (decay['Time_Sec'].iloc[-1] - decay['Time_Sec'].iloc[0])
        norm_auc = auc / max_area if max_area > 0 else 0
        
        stats.append({
            'Signal': sig, 'Baseline_r': baseline_r,
            'Half_Life': t_half, 'Persistence_Score': norm_auc
        })
    return pd.DataFrame(stats)

### Main Analysis

In [5]:
pc_subs  = utils.process_analysis.get_subjects_by_condition(CONDITIONS_FILE, target_condition = 'PC',  exclude_prior_exposure=False)
pcm_subs = utils.process_analysis.get_subjects_by_condition(CONDITIONS_FILE, target_condition = 'PCM', exclude_prior_exposure=False)

# --- MAIN LOOP ---
print("Running Sliding Window Analysis...")
timecourse_results = []

for signal in signals:
    print(f"  Processing {signal}...")
    try:
        d_pc  = utils.process_analysis.load_physio_data(DATA_DIR, pc_subs, signal)
        d_pcm = utils.process_analysis.load_physio_data(DATA_DIR, pcm_subs, signal)
        
        # Run Sliding Window
        df_res = sliding_window_group_isc(d_pc, d_pcm, WIN_SIZE, STEP_SIZE, n_boot=200)
        df_res['Signal'] = signal
        timecourse_results.append(df_res)
    except Exception as e:
        print(f"    Error: {e}")

# Save Timecourses
df_final = pd.concat(timecourse_results, ignore_index=True)
df_final.to_csv(os.path.join(OUTPUT_DIR, "decay_timecourses.csv"), index=False)

# Calculate and Save Metrics
print("Calculating Decay Metrics...")
df_metrics = quantify_decay(df_final)
df_metrics.to_csv(os.path.join(OUTPUT_DIR, "decay_metrics.csv"), index=False)

print("Done.")

Running Sliding Window Analysis...
  Processing Gaze_X...
  Processing Gaze_Y...
  Processing Pupil_Mean...
  Processing Heart_Rate...
  Processing GSR...
Calculating Decay Metrics...
Done.
